<h1 align="center">CPSC 4830</h1>
<h2 align="center">MIDTERM 1</h2>
<h2 align="center">TIME - 4 HRS</h2>
<h2 align="center">MAX MARKS - 40</h2>

<h3 align="center">All work must be completed in this file and Submit the resulting .ipynb in D2L</h3>

## QUESTION 1 [20 Marks]

## Introduction

Bigmart is a big supermarket chain, with stores all around the country. The management of the shop had set out a challenge to all Data Scientist to help them create a model that can predict the sales per product for each store.

The shop has collected sales data of products across 10 stores in different cities over a given period of time.

## Breakdown of the Problem Statement

This is a supervised machine learning problem with a target label as : (Item_Outlet_Sales)

Also since we are expected to predict the sale price for a given product, it becomes a regression task. There are 10 feature columns. Hence we expect you to use dimension reduction approach along with regression model for prediction. Select optimal number of principal component using explained variance measure. Hypertuning is crucial and should be performed to optimize the model.

EDA and inferential analysis is very important. Please provide appropriate reasons of each steps. Provide conclusion and future steps as a closure.

## QUESTION 2

### The SMS Spam Collection is a set of SMS tagged messages that have been collected from SMS Spam research database. It contains one set of SMS messages in English of 5,572 messages, tagged acording being ham (legitimate) or spam.

### Out of the 5572 sets of sms messages, approximately 20% of the messages have been trimmed and kept aside for Evaluating your model by the Instructor. You only have 4458 rows to build your model. Rest will be used by the instructor as a blind evaluation of your model. This data is not provided to you.

### Use this dataset to build a prediction model as follows that will accurately classify which texts are spam?

### 1. Use Count vectorizer to convert the texts into numerical values. Make sure to perform EDA first to check for any issues with the data.  **[5 Marks]**

### 2. Use Logistic Regression with L2 Regularisation (using your intelligent choice of Hyperparameters), find the accuracy/precison/recall of SPAM/HAM detection. Analyze your results and provide what could improve your results. **[10 Marks]**

### 3. Compare your result with Random Forest Classifier for this same dataset. (using your intelligent choice of Hyperparameters).**[5 Marks]**

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)

%matplotlib inline

df = pd.read_csv("spamham.csv")  # replace with actual file
df.head()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4457 entries, 0 to 4456
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  4457 non-null   object
 1   Message   4457 non-null   object
dtypes: object(2)
memory usage: 69.8+ KB


,Category,Message
count,4457,4457
unique,2,4157
top,ham,"Sorry, I'll call later"
freq,3863,26


In [19]:
col_map = {c.lower(): c for c in df.columns}
label_col  = col_map.get("category", col_map.get("v1", list(df.columns)[0]))
text_col   = col_map.get("message",  col_map.get("v2", list(df.columns)[1]))

In [20]:
RANDOM_STATE = 42
DATA_PATH = "spamham.csv"

df = df[[label_col, text_col]].dropna().drop_duplicates()
df.columns = ["label", "text"]

print("\n=== EDA ===")
print("Shape:", df.shape)
print("Missing total:", df.isna().sum().sum())
print("Duplicates:", df.duplicated().sum())
print("Class balance:\n", df["label"].value_counts())
df["len"] = df["text"].astype(str).str.len()
print("\nLength stats:\n", df["len"].describe())

# ============== 1) Train / test split ==============
y = df["label"].str.lower().map({"ham":0, "spam":1}).astype(int).values
X = df["text"].astype(str).values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)


=== EDA ===
Shape: (4157, 2)
Missing total: 0
Duplicates: 0
Class balance:
 label
ham     3635
spam     522
Name: count, dtype: int64

Length stats:
 count    4157.000000
mean       79.432523
std        59.304106
min         2.000000
25%        36.000000
50%        61.000000
75%       119.000000
max       910.000000
Name: len, dtype: float64


In [21]:
countvec = CountVectorizer(
    stop_words="english",
    ngram_range=(1,2),  # unigrams + bigrams help on this dataset
    min_df=2,
    binary=True,        # binary features work nicely for LR with SMS
    max_features=3000   # cap dimensionality (also helps RF)
)

In [22]:
X_train_vec = countvec.fit_transform(X_train)
X_test_vec  = countvec.transform(X_test)

In [23]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
lr = LogisticRegression(
    penalty="l2",
    class_weight="balanced",
    max_iter=1500,
    random_state=RANDOM_STATE
)
param_grid_lr = {"C": [0.5, 1.0, 3.0, 10.0]}
gs_lr = GridSearchCV(lr, param_grid_lr, scoring="f1", cv=cv, n_jobs=-1, verbose=0)
gs_lr.fit(X_train_vec, y_train)

lr_best = gs_lr.best_estimator_
y_pred_lr = lr_best.predict(X_test_vec)

acc_lr = accuracy_score(y_test, y_pred_lr)
prec_lr, rec_lr, f1_lr, _ = precision_recall_fscore_support(
    y_test, y_pred_lr, average="binary", zero_division=0
)
cm_lr = confusion_matrix(y_test, y_pred_lr)
print("\n=== Logistic Regression (L2, CountVectorizer) ===")
print("Best params:", gs_lr.best_params_)
print(f"Accuracy: {acc_lr:.4f} | Precision(spam): {prec_lr:.4f} | Recall(spam): {rec_lr:.4f} | F1(spam): {f1_lr:.4f}")
print("Confusion matrix (rows=true, cols=pred):\n", pd.DataFrame(cm_lr,
      index=["True ham","True spam"], columns=["Pred ham","Pred spam"]))
print("\nClassification report:\n",
      classification_report(y_test, y_pred_lr, target_names=["ham","spam"], zero_division=0))



=== Logistic Regression (L2, CountVectorizer) ===
Best params: {'C': 0.5}
Accuracy: 0.9844 | Precision(spam): 0.9596 | Recall(spam): 0.9135 | F1(spam): 0.9360
Confusion matrix (rows=true, cols=pred):
            Pred ham  Pred spam
True ham        724          4
True spam         9         95

Classification report:
               precision    recall  f1-score   support

         ham       0.99      0.99      0.99       728
        spam       0.96      0.91      0.94       104

    accuracy                           0.98       832
   macro avg       0.97      0.95      0.96       832
weighted avg       0.98      0.98      0.98       832



In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import numpy as np
import pandas as pd

# 1) Dense matrices (RF handles dense better for many features)
X_train_dense = X_train_vec.toarray()
X_test_dense  = X_test_vec.toarray()

# 2) CV + compact hyperparameter grid (good tradeoff for this dataset size)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
rf = RandomForestClassifier(random_state=42, n_jobs=3, class_weight="balanced",
                            max_samples=0.8,       # subsample each tree to reduce cost
                            bootstrap=True)

param_grid_rf = {
    "n_estimators": [150, 250],         # more trees → stabler metrics
    "max_depth": [None, 40],        # None lets trees grow fully; 30/60 control overfitting
    "min_samples_split": [2],        # regularization
    "min_samples_leaf": [1, 2],         # regularization
    "max_features": ["sqrt", 0.25]      # feature subsampling (sqrt is standard)
}

gs_rf = GridSearchCV(rf, param_grid_rf, scoring="f1", cv=cv, n_jobs=3, verbose=0)
gs_rf.fit(X_train_dense, y_train)

rf_best = gs_rf.best_estimator_
y_pred_rf = rf_best.predict(X_test_dense)

acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf, rec_rf, f1_rf, _ = precision_recall_fscore_support(
    y_test, y_pred_rf, average="binary", zero_division=0
)
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("\n=== Random Forest (CountVectorizer) ===")
print("Best params:", gs_rf.best_params_)
print(f"Accuracy: {acc_rf:.4f} | Precision(spam): {prec_rf:.4f} | Recall(spam): {rec_rf:.4f} | F1(spam): {f1_rf:.4f}")
print("Confusion matrix (rows=true, cols=pred):\n", pd.DataFrame(
      cm_rf, index=['True ham','True spam'], columns=['Pred ham','Pred spam']))
print("\nClassification report:\n",
      classification_report(y_test, y_pred_rf, target_names=["ham","spam"], zero_division=0))

# (Optional) quick side-by-side if you still have lr metrics from Point 2 in variables:
try:
    comp = pd.DataFrame([
        ["Logistic Regression (L2)", acc_lr, prec_lr, rec_lr, f1_lr],
        ["Random Forest",            acc_rf, prec_rf, rec_rf, f1_rf]
    ], columns=["Model","Accuracy","Precision(spam)","Recall(spam)","F1(spam)"])
    print("\n=== Model comparison ===\n", comp.to_string(index=False))
except NameError:
    pass

NameError: name 'X_train_vec' is not defined